# Baseline vs fine-tuned evaluation

This notebook re-evaluates the two tracked custom YOLOv8 weights on a supplied dataset. Set `HELMET_DATA_YAML` to the same Version 4 test configuration used for the intended comparison. The full dataset and original validation logs are not stored in Git.

The project-reported metrics are shown separately from any fresh run so they are not mistaken for newly computed results.

In [ ]:
import csv
import os
from pathlib import Path
from ultralytics import YOLO

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "config" / "paths.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook inside the project checkout.")
DATA_VALUE = os.environ.get("HELMET_DATA_YAML")
if not DATA_VALUE:
    raise RuntimeError("Set HELMET_DATA_YAML to a compatible Version 4 data.yaml.")
DATA_YAML = Path(DATA_VALUE).expanduser().resolve()
if not DATA_YAML.is_file():
    raise FileNotFoundError(DATA_YAML)

WEIGHTS = {
    "baseline": PROJECT_ROOT / "model" / "archive" / "helmet_best_old.pt",
    "fine_tuned": PROJECT_ROOT / "model" / "helmet_best.pt",
}
for role, path in WEIGHTS.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing {role} checkpoint: {path}")
print(f"Evaluation dataset: {DATA_YAML}")

## Run validation

Ultralytics reports standard aggregate metrics and per-class box metrics. Validation artifacts are written to `runs/helmet_validation/`; the script does not alter model weights.

In [ ]:
runs = {}
for role, weights_path in WEIGHTS.items():
    model = YOLO(str(weights_path))
    metrics = model.val(
        data=str(DATA_YAML),
        imgsz=640,
        project=str(PROJECT_ROOT / "runs" / "helmet_validation"),
        name=role,
    )
    box = metrics.box
    runs[role] = {
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP@50": float(box.map50),
        "mAP@50:95": float(box.map),
    }
    print(f"\n{role}: {runs[role]}")

    class_ids = list(getattr(box, "ap_class_index", []))
    class_names = model.names
    per_class = []
    for position, class_id in enumerate(class_ids):
        name = class_names.get(int(class_id), str(class_id)) if isinstance(class_names, dict) else class_names[int(class_id)]
        per_class.append((
            name, float(box.p[position]), float(box.r[position]),
            float(box.ap50[position]), float(box.ap[position].mean()),
        ))
    print("per-class: (class, precision, recall, mAP@50, mAP@50:95)")
    for row in per_class:
        print(row)

## Recorded project metrics

The following cell reads the documented, project-reported comparison CSV. Compare these values with the fresh `runs` results only after confirming dataset version and split match.

In [ ]:
metrics_csv = PROJECT_ROOT / "results" / "metrics" / "comparison.csv"
with metrics_csv.open(newline="", encoding="utf-8") as stream:
    reported = list(csv.DictReader(stream))
print(f"Project-reported metrics from {metrics_csv.relative_to(PROJECT_ROOT)}")
for row in reported:
    print(f"{row['metric']:<22} baseline={float(row['baseline']):.3f}  fine_tuned={float(row['fine_tuned']):.3f}")